In [2]:
import numpy as np
import math
import rclpy
import threading
import time

from semantic_digital_twin.world import World
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.robots.soft_trunk import SoftTrunk


/home/rinaalo/.virtualenvs/cram-env/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("soft_pcc_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

In [ ]:
world = World()
trunk, k_dof, p_dof = SoftTrunk.build_pcc(world, 15)

In [5]:
# Setup Visualization
tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

print(f"Robot '{trunk.name.name}' is ready. Check RViz.")

Robot 'soft_trunk' is ready. Check RViz.


In [6]:
# Animation Loop: Bending the Soft Robot
try:
    while True:
        print("Animating PCC: Bending and Rotating...")
        # T alternates bending, t alternates the plane of bending
        for t in np.linspace(0, 10, 200):
            # simulate a bending motion (kappa) and a rotating motion (phi)
            # Curvature: 0 (straight) to 4 (tight bend)
            k_val = 2.0 * (np.sin(t) + 1.0) 
            # Bending plane: 0 to 360 degrees
            p_val = t 
            
            # Update the state values in the world
            world.state[k_dof.id].position = float(k_val)
            world.state[p_dof.id].position = float(p_val)
            
            # Recompute Forward Kinematics
            world.notify_state_change()
            
            # Update RViz
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.05)
except KeyboardInterrupt:
    print("Animation stopped.")

Animating PCC: Bending and Rotating...
Animating PCC: Bending and Rotating...
Animating PCC: Bending and Rotating...
Animating PCC: Bending and Rotating...
Animation stopped.


In [7]:
node.destroy_node()
rclpy.shutdown()